# Final Project Phase 2: AI Agent-Powered Automation for Peloton's Fitness Ecosystem
# Shishir Deshpande, MSDS 442

### Importing required libraries

In [ ]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)

from dotenv import load_dotenv

load_dotenv() # this will read my secrets and API keys from the .env file
os.environ["USER_AGENT"] = "MSDS442-Phase2Project-Deshpande"

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
import chromadb
from langchain_core.documents import Document

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END

from typing import TypedDict, Literal
from IPython.display import display, Markdown, Image
model = ChatOpenAI(model="gpt-4o-mini")

### Listing of files used for Requirement 1
|File|Description|Used In|
|---|---|---|
|Analysis Code (Requirements 2, 3 & 4).ipynb|Generated user stories + training/testing data via GPT-4o-mini|Req 2, 3, 4|
|Requirement 4_TrainingTestingData.csv|45-row training/testing dataset, 5 agents, 15 user stories	|Req 3, 4|
|Requirements_Specification.pdf | Peloton agent responsibilities and project scope | Req 1, 2|
|Subset of Requirement 4 CSV (6 training rows/agent; testing rows held out to prevent data leakage)|Training-only knowledge base for RAG retrieval|Req 4|

#### Loading files from CSV

In [ ]:
import pandas as pd

csv_path = r"Requirement 4_TrainingTestingData.csv"

df = pd.read_csv(csv_path, encoding='utf-8-sig')

print(f"Loaded {len(df)} rows")
print(df.columns.tolist())
print()
print(df["AI Agent"].value_counts())

#### Building per-agent knowledge base documents

In [ ]:

# Mapping csv agent labels to short keys
agent_key_map = {
    "Business/Marketing AI Agent": "business",
    "Data Science AI Agent": "data_science",
    "Membership/Fraud Detection AI Agent": "membership",
    "Order/Shipping AI Agent": "order",
    "Product Recommendation AI Agent": "product_rec"
}

# Building agent-wise knowledge base docs from Sample Query + Expected Response
agent_documents = {key: [] for key in agent_key_map.values()}

for _, row  in df.iterrows():
    if row["Data Type"].strip() != "Training":
        continue # skip testing data rows
    key = agent_key_map[row["AI Agent"]]
    content = f"Q: {row['Sample Query']}\nA: {row['Expected Response']}"
    agent_documents[key].append(
        Document(page_content=content, metadata={"user_story": row["User-Story"], "data_type": row["Data Type"]})
    )

for key, docs in agent_documents.items():
    print(f"{key}: {len(docs)} documents")

#### Setting up embeddings for each agent type

In [ ]:
embeddings = OpenAIEmbeddings()

chroma_client = chromadb.Client()

agent_vectorstores = {}

In [ ]:
for key, docs in agent_documents.items():
    collection_name = f"peloton_{key}"

    # Deleting any existing collection with this name to prevent duplicate indexing on rerun
    try:
        chroma_client.delete_collection(collection_name)
    except Exception:
        pass  # collection didn't exist yet -- fine

    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        collection_name=collection_name,
        client=chroma_client
    )
    agent_vectorstores[key] = vectorstore

    actual_count = vectorstore._collection.count()
    expected_count = len(docs)
    status = "OK" if actual_count == expected_count else "MISMATCH"
    print(f"{key}: {actual_count} embedded (expected {expected_count}) -- {status}")

#### Defining the agent node functions and router

In [ ]:
# Defining the class
class PelotonState(TypedDict):
    user_query: str
    agent_type: Literal["business", "data_science", "membership", "order", "product_rec"]
    response: str

# Router to classify user query to one of the agent types
def router_node(state: PelotonState) -> PelotonState:
    query = state["user_query"]
    router_prompt = f"""Classify this Peloton customer/employee query into exactly one category. You must respond with ONLY one word: business, data_science, membership, order, or product_rec.

    Categories are as follows:
    - business: marketing campaigns, ROI, brand performance, content strategy
    - data_science: fitness metrics, heart rate, VO2 max, performance data, algorithms
    - membership: password resets, billing issues, account security, membership tiers
    - order: order status, shipping, delivery, warranty, returns
    - product_rec: product recommendations, accessories, compatibility, trending items

    Query: {query}"""
    response = model.invoke([HumanMessage(content=router_prompt)])
    agent_type = response.content.strip().lower()
    if agent_type not in ["business", "data_science", "membership", "order", "product_rec"]: # this is our guardrail to not allow invalid agent invokes
        agent_type = "business" # fallback to default agent
    return {**state, "agent_type": agent_type}

def route_decision(state: PelotonState) -> str:
    return state["agent_type"]

# Defining a generic agent node builder
def make_agent_node(agent_key: str, agent_label: str):
    def agent_node(state: PelotonState) -> PelotonState:
        query = state["user_query"]
        retriever = agent_vectorstores[agent_key].as_retriever(search_kwargs = {"k":3})
        relevant_docs = retriever.invoke(query)
        context = "\n\n".join([doc.page_content for doc in relevant_docs])

        system_prompt = f"""You are the {agent_label} for Peloton. Use the reference Q&A pairs detailed below to answer the user's query in a helpful tone that is consistent with Peloton's brand. If the reference material doesn't directly answer or cover the query, please use your best judgment while staying on-brand. 
        
        Reference material:
        {context}"""

        response = model.invoke([SystemMessage(content = system_prompt), HumanMessage(content = query)])
        return {**state, "response":response.content}
    return agent_node


# Agent 1: Business/Marketing AI agent
## This agent can answer questions on marketing campaigns, ROI, and other business topics
business_agent_node = make_agent_node("business", "Business/Marketing AI Agent")

# Agent 2: Data Science AI agent
## This agent can answer questions on fitness metrics, heart rate, and other analytics
data_science_agent_node = make_agent_node("data_science", "Data Science AI Agent")

# Agent 3: Membership/Fraud Detection AI Agent
## This agent can answer questions on billing issues, fraudulent activity, and passwords
membership_agent_node = make_agent_node("membership", "Membership/Fraud Detection AI Agent")

# Agent 4: Order/Shipping AI agent
## This agent can answer questions on delays, delivery methods, warranty etc.
order_agent_node = make_agent_node("order", "Order/Shipping AI Agent")

# Agent 5: Product Recommendation AI agent
## This agent can answer questions on new products to buy, accessories etc. 
product_rec_agent_node = make_agent_node("product_rec", "Product Recommendation AI Agent")

# Confirming agent builds
print("All agents built successfully.")

### LangGraph Architecture for Requirement 2

In [ ]:
# Setting up the nodes to connect the router to the 5 agents
graph = StateGraph(PelotonState)
graph.add_node("router", router_node)
graph.add_node("business", business_agent_node)
graph.add_node("data_science", data_science_agent_node)
graph.add_node("membership", membership_agent_node)
graph.add_node("order", order_agent_node)
graph.add_node("product_rec", product_rec_agent_node)

graph.set_entry_point("router")
graph.add_conditional_edges("router", route_decision, {
    "business": "business",
    "data_science": "data_science",
    "membership": "membership",
    "order": "order",
    "product_rec": "product_rec"
})

# Compiling the graph
app = graph.compile()

# Displaying the graph as a mermaid diagram
display(Image(app.get_graph().draw_mermaid_png()))
print(app.get_graph().draw_mermaid())

In [ ]:
# Selecting target use stories using a substring match
selected_user_story_keywords = {
    "business":  "Summer Fitness Challenge",
    "data_science": "heart rate",
    "membership": "reset my account password",
    "order": "delayed order",
    "product_rec": "complementary products"
}

demo_queries = {}
demo_expected = {}
demo_user_stories = {}

for key, keyword in selected_user_story_keywords.items():
    match = df[(df['User-Story'].str.contains(keyword, case= False, na=False)) & (df["Data Type"].str.strip() == "Testing")]
    if len(match) == 0:
        print(f"No matching test query found for {keyword}")
        continue
    row = match.iloc[0]
    demo_queries[key] = row["Sample Query"]
    demo_expected[key] = row["Expected Response"]
    demo_user_stories[key] = row["User-Story"]
    print(f"{key}: {row['Sample Query']}")

### Demo for Requirement 4

In [ ]:
demo_results = {}

for expected_agent, query in demo_queries.items():
    result = app.invoke({"user_query": query, "agent_type": "", "response": ""})
    demo_results[expected_agent] = result
    routed_correctly = "Successfully" if result["agent_type"] == expected_agent else "Not successfully"

    md_output = f"""---
**{routed_correctly} routed to:** `{result['agent_type']}` (expected: `{expected_agent}`)

**User Story:** {demo_user_stories[expected_agent]}

**Query:** {query}

**Response:** {result['response']}
"""
    display(Markdown(md_output))